# Agent 9 â€” Greenwashing Detection

**What this notebook does:**  
Imports the 8-Test greenwashing results produced by Claude Projects for each company, calculates a greenwashing risk score, and applies the exclusion and watchlist rules from the mandate.

**How to present this to investors:**  
> *Our greenwashing agent applies a systematic 8-dimension forensic analysis to every company's sustainability claims â€” assessing specificity, numeric backing, baseline comparisons, target credibility, time horizons, scope of coverage, external verification, and capex consistency. Companies scoring HIGH risk on three or more dimensions are excluded from the portfolio. This process converts subjective ESG claims into a structured, auditable risk score.*

**How this workflow operates:**  
1. RAG Operator runs the standard 8-Test prompt in each Claude Project  
2. RAG Operator saves the JSON output as `outputs/rag/greenwash_TICKER.json`  
3. This notebook imports all JSON files, scores them, and applies exclusion rules  

**The 8 dimensions assessed:**  
Specificity | Metric | Baseline | Target | Time Horizon | Scope | Verification | Consistency

**Run after:** `03_esg_scoring.ipynb` and `06_document_intelligence.ipynb`

## Step 1 â€” The standard 8-Test RAG prompt

Copy this prompt into each Claude Project. Replace [COMPANY] with the company name.  
Save the JSON output as `outputs/rag/greenwash_TICKER.json`.

In [1]:
GREENWASH_PROMPT = """
You are an ESG forensic analyst. For [COMPANY], analyse the most recent sustainability 
report and assess each of the 8 greenwashing dimensions below.

For each dimension provide:
  (a) direct quote with page number
  (b) numerical value or factual statement
  (c) red-flag rating: LOW / MED / HIGH / MISSING
  (d) one to two sentences of reasoning

If a dimension has no information, mark it MISSING â€” never invent.

Output ONLY valid JSON with exactly this structure:
{
  "ticker": "...",
  "company_name": "...",
  "extraction_date": "YYYY-MM-DD",
  "analyst_note": "...",
  "dimensions": {
    "specificity":   {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "metric":        {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "baseline":      {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "target":        {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "time_horizon":  {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "scope":         {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "verification":  {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null},
    "consistency":   {"quote": null, "page": null, "value": null, "rating": null, "reasoning": null}
  }
}

Red-flag guidance:
  specificity  â€” LOW if exact wording; HIGH if vague terms only
  metric       â€” LOW if numeric backing; HIGH if no numbers
  baseline     â€” LOW if comparison reference present; HIGH if absent or cherry-picked
  target       â€” LOW if binding commitment; HIGH if non-binding or missing
  time_horizon â€” LOW if near-term date; HIGH if 2050+ only and unverifiable
  scope        â€” LOW if clear coverage; HIGH if ambiguous which division/asset
  verification â€” LOW if externally assured; HIGH if self-reported only
  consistency  â€” LOW if capex/lobbying matches claims; HIGH if contradiction
"""

print("Standard 8-Test greenwashing prompt ready.")
print("")
print("INSTRUCTIONS FOR RAG OPERATOR:")
print("1. Open a Claude Project for the company you are analysing")
print("2. Make sure the company's sustainability report PDF is uploaded")
print("3. Replace [COMPANY] with the actual company name")
print("4. Paste the prompt and run it")
print("5. Copy the JSON output")
print("6. Save it as:  outputs/rag/greenwash_TICKER.json")
print("   Example:     outputs/rag/greenwash_ASML.json")

Standard 8-Test greenwashing prompt ready.

INSTRUCTIONS FOR RAG OPERATOR:
1. Open a Claude Project for the company you are analysing
2. Make sure the company's sustainability report PDF is uploaded
3. Replace [COMPANY] with the actual company name
4. Paste the prompt and run it
5. Copy the JSON output
6. Save it as:  outputs/rag/greenwash_TICKER.json
   Example:     outputs/rag/greenwash_ASML.json


## Step 2 â€” Create template and output folder

In [2]:
import os
import json
import pandas as pd
import numpy as np
import glob
from datetime import date

TODAY = str(date.today())
RAG_DIR = "../outputs/rag"
os.makedirs(RAG_DIR, exist_ok=True)

# Sample template showing the expected JSON format
sample = {
  "ticker": "EXAMPLE",
  "company_name": "Example Company SA",
  "extraction_date": TODAY,
  "analyst_note": "Generally transparent reporting; verification gaps on Scope 3.",
  "dimensions": {
    "specificity":  {"quote": "We will become carbon neutral in all operations", "page": 4, "value": "carbon neutral", "rating": "MED", "reasoning": "Uses broad term 'carbon neutral' without defining Scope coverage or offsets."},
    "metric":       {"quote": "Absolute Scope 1+2 reduction of 46% by 2030 vs 2019 baseline", "page": 12, "value": "46%", "rating": "LOW", "reasoning": "Clear numeric target with year and baseline."},
    "baseline":     {"quote": "vs 2019 baseline of 2.4 MtCO2e", "page": 12, "value": "2019 = 2.4 Mt", "rating": "LOW", "reasoning": "Specific baseline year and value provided."},
    "target":       {"quote": "SBTi-validated target submitted March 2023", "page": 8,  "value": "SBTi validated", "rating": "LOW", "reasoning": "Third-party validated â€” binding commitment."},
    "time_horizon": {"quote": "by 2030 and net zero by 2050", "page": 5, "value": "2030 / 2050", "rating": "MED", "reasoning": "Near-term 2030 target is credible; 2050 net zero is long-dated and less verifiable."},
    "scope":        {"quote": "covering Scope 1, 2, and material Scope 3 categories", "page": 12, "value": "S1+2+3", "rating": "LOW", "reasoning": "Scope 3 included though 'material categories' leaves some ambiguity."},
    "verification": {"quote": "Assured by EY to limited assurance (ISAE 3000)", "page": 95, "value": "EY limited", "rating": "MED", "reasoning": "Limited assurance is weaker than reasonable assurance â€” upgrade expected under CSRD."},
    "consistency":  {"quote": None, "page": None, "value": "No contradiction found", "rating": "LOW", "reasoning": "Capex allocation (42% green) consistent with stated transition commitments."}
  }
}

template_path = f"{RAG_DIR}/greenwash_EXAMPLE_TEMPLATE.json"
with open(template_path, "w") as f:
    json.dump(sample, f, indent=2)

print(f"Template saved: {template_path}")
print(f"RAG output folder: {RAG_DIR}")

Template saved: ../outputs/rag/greenwash_EXAMPLE_TEMPLATE.json
RAG output folder: ../outputs/rag


## Step 3 — Import from RAG Screening Sheet (Excel)

Reads the filled-in RAG Screening Sheet workbook and converts each row
into the standard greenwashing JSON format.  
If no Excel data exists yet, this cell is skipped and JSON files are loaded directly in Step 4.

In [ ]:
# ── Pre-flight validation ──────────────────────────────────────────────────────
# The pipeline scores the greenwash_*.json files in outputs/rag/ (loaded in
# Step 4). This cell reports how well those JSONs cover the final portfolio, and
# notes the RAG Screening Sheet workbook if present. It is advisory under
# VALIDATION_MODE = "warn"; set "strict" to raise on gaps before final submission.
#
# VALIDATION_MODE:  "warn" (default) | "strict" | "skip"
VALIDATION_MODE = "warn"

import os, glob, json
import pandas as pd

EXCEL_PATH = "../data/rag/RAG_Screening_Sheet_Workbook_v1.xlsx"
RAG_DIR    = "../outputs/rag"
errors, warnings = [], []

# ── 1. Final portfolio holdings ────────────────────────────────────────────────
port_files = sorted(glob.glob("../outputs/portfolio/final_portfolio_*.csv"))
if not port_files:
    errors.append("CRITICAL: no final_portfolio_*.csv — run notebook 10 first")
    portfolio_tickers = set()
else:
    _port = pd.read_csv(port_files[-1])
    # The reworked final portfolio is keyed on yf_ticker (no Bloomberg 'ticker').
    _tcol = next((c for c in ["yf_ticker", "ticker"] if c in _port.columns), None)
    portfolio_tickers = set(_port[_tcol].astype(str).str.strip()) if _tcol else set()
    print(f"Final portfolio: {len(_port)} holdings  ({os.path.basename(port_files[-1])}, key='{_tcol}')")

# ── 2. Greenwash 8-Test JSON coverage ──────────────────────────────────────────
gw_json = [f for f in glob.glob(f"{RAG_DIR}/greenwash_*.json") if "TEMPLATE" not in f]
gw_tickers = set()
for f in gw_json:
    try:
        gw_tickers.add(str(json.load(open(f)).get("ticker", "")).strip())
    except Exception as exc:
        warnings.append(f"could not read {os.path.basename(f)}: {exc}")
covered   = portfolio_tickers & gw_tickers
uncovered = portfolio_tickers - gw_tickers
print(f"Greenwash 8-Test JSONs present: {len(gw_json)}  |  cover {len(covered)}/{len(portfolio_tickers)} holdings")
if uncovered:
    warnings.append(f"{len(uncovered)} holding(s) without an 8-Test JSON yet: {sorted(uncovered)}")

# ── 3. RAG Screening Sheet workbook (optional — the JSON path is primary) ──────
if os.path.exists(EXCEL_PATH):
    print(f"RAG Screening Sheet workbook present: {EXCEL_PATH}")
else:
    warnings.append("RAG Screening Sheet workbook not found (optional — JSON path is used)")

# ── 4. Report ──────────────────────────────────────────────────────────────────
print("=" * 65)
print("PRE-FLIGHT VALIDATION — Notebook 09 Greenwashing")
print("=" * 65)
for e in errors:
    print(f"  ERROR    {e}")
for w in warnings:
    print(f"  WARNING  {w}")
if not errors:
    print("  OK — continuing (warnings above are advisory)")
print("=" * 65)

if VALIDATION_MODE == "strict" and (errors or uncovered):
    raise RuntimeError(f"Validation failed — {len(errors)} error(s); "
                       f"{len(uncovered)} holding(s) without an 8-Test JSON.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Step 3 — Derive the greenwashing JSONs FROM the RAG Screening Sheet workbook
# ════════════════════════════════════════════════════════════════════════════
# HOW_TO_USE.txt: the workbook is the authoritative human-reviewed layer; the
# greenwash_*.json files are DERIVED from it. This cell reads the populated
# 8-Test rows, resolves each to its Yahoo ticker by ISIN (the workbook's own
# Ticker column is Bloomberg-style — the rest of the pipeline keys on yf_ticker),
# translates PASS/PARTIAL/FAIL/MISSING -> LOW/MED/HIGH/MISSING, and writes one
# greenwash_<yf_ticker>.json per holding. Step 4 then scores those files.
import glob, os, json
import pandas as pd
from datetime import date

EXCEL_PATH = "../data/rag/RAG_Screening_Sheet_Workbook_v1.xlsx"
RAG_DIR    = "../outputs/rag"
os.makedirs(RAG_DIR, exist_ok=True)
TODAY = str(date.today())

# ISIN -> Yahoo ticker for the 20 portfolio holdings (workbook rows are matched
# on ISIN; only these rows are turned into scored JSONs).
ISIN_TO_YF = {
    "CH0011075394": "ZURN.SW",  "NL0000360618": "SBMO.AS",  "CH0010675863": "SPSN.SW",
    "CH0012221716": "ABBN.SW",  "FR0000121964": "LI.PA",    "ES0105025003": "MRL.MC",
    "CH0360674466": "GALE.SW",  "DE000ENAG999": "EOAN.DE",  "IE00BF0L3536": "A5G.IR",
    "BE0003739530": "UCB.BR",   "GB0009895292": "AZN.L",    "NL0000303709": "AGN.AS",
    "GB0008706128": "LLOY.L",   "SE0000695876": "ALFA.ST",  "NO0005052605": "NHY.OL",
    "SE0005190238": "TEL2-B.ST","ES0148396007": "ITX.MC",   "FI0009014377": "ORNBV.HE",
    "SE0000872095": "SOBI.ST",  "LU0075646355": "SUBC.OL",
}
# ISIN -> company name. The workbook's Company Name column is a formula; openpyxl
# drops cached formula values on save, so the name is resolved from this map.
ISIN_TO_NAME = {
    "CH0011075394": "Zurich Insurance Group Ltd", "NL0000360618": "SBM Offshore NV",
    "CH0010675863": "Swiss Prime Site AG", "CH0012221716": "ABB Ltd.",
    "FR0000121964": "Klepierre SA", "ES0105025003": "MERLIN Properties SOCIMI, S.A.",
    "CH0360674466": "Galenica AG", "DE000ENAG999": "E.ON SE",
    "IE00BF0L3536": "AIB Group plc", "BE0003739530": "UCB S.A.",
    "GB0009895292": "AstraZeneca PLC", "NL0000303709": "Aegon Ltd.",
    "GB0008706128": "Lloyds Banking Group plc", "SE0000695876": "Alfa Laval AB",
    "NO0005052605": "Norsk Hydro ASA", "SE0005190238": "Tele2 AB Class B",
    "ES0148396007": "Industria de Diseno Textil, S.A.", "FI0009014377": "Orion Oyj Class B",
    "SE0000872095": "Swedish Orphan Biovitrum AB", "LU0075646355": "Subsea 7 S.A.",
}

# 8-Test column label -> JSON dimension key
DIM_COL_MAP = {
    "8-Test 1: Specificity":  "specificity", "8-Test 2: Metric":       "metric",
    "8-Test 3: Baseline":     "baseline",    "8-Test 4: Target":       "target",
    "8-Test 5: Time Horizon": "time_horizon","8-Test 6: Scope":        "scope",
    "8-Test 7: Verification": "verification","8-Test 8: Consistency":  "consistency",
}
# Workbook vocabulary -> pipeline vocabulary
RATING_TRANSLATE = {"PASS": "LOW", "PARTIAL": "MED", "FAIL": "HIGH", "MISSING": "MISSING"}

excel_generated = 0

if os.path.exists(EXCEL_PATH):
    xl = pd.ExcelFile(EXCEL_PATH)
    if "RAG Screening Sheet" in xl.sheet_names:
        # Header on row 6 (0-indexed 5); rows 1-5 are banners/auto-numbers/formulas.
        rag_df = pd.read_excel(EXCEL_PATH, sheet_name="RAG Screening Sheet", header=5)
        rag_df.columns = [str(c).strip() for c in rag_df.columns]

        for _, row in rag_df.iterrows():
            isin = str(row.get("ISIN", "")).strip().upper()
            yf_ticker = ISIN_TO_YF.get(isin)
            if not yf_ticker:                       # not a portfolio holding
                continue

            # 8-Test ratings — one quote/page per company (the workbook tests one
            # main claim across 8 dimensions; HOW_TO_USE.txt's model).
            key_quote = row.get("Key Evidence Quote", "")
            key_page  = row.get("Key Evidence Page Ref", "")
            dimensions = {}
            for col_label, dim_key in DIM_COL_MAP.items():
                raw = str(row.get(col_label, "MISSING")).strip().upper()
                if raw in ("", "NAN", "NONE"):
                    raw = "MISSING"
                dimensions[dim_key] = {
                    "quote":     "" if pd.isna(key_quote) else key_quote,
                    "page":      "" if pd.isna(key_page)  else key_page,
                    "value":     "",
                    "rating":    RATING_TRANSLATE.get(raw, raw),
                    "reasoning": "",
                }

            # Skip rows with no real ratings (workbook row not yet filled)
            if not [d["rating"] for d in dimensions.values() if d["rating"] != "MISSING"]:
                continue

            out = {
                "ticker":          yf_ticker,
                "company_name":    ISIN_TO_NAME.get(isin, yf_ticker),
                "extraction_date": TODAY,
                "analyst_note":    str(row.get("Verdict Rationale", "")),
                "dimensions":      dimensions,
            }
            with open(f"{RAG_DIR}/greenwash_{yf_ticker}.json", "w") as f:
                json.dump(out, f, indent=2)
            excel_generated += 1

print(f"JSON files derived from the RAG workbook: {excel_generated}")
print(f"Total JSON files in {RAG_DIR}: "
      f"{len([f for f in glob.glob(RAG_DIR+'/*.json') if 'TEMPLATE' not in f])}")
if excel_generated == 0:
    print("  (workbook not populated yet — Step 4 will score any JSONs already present)")


## Step 4 — Import all greenwashing JSON files

In [5]:
# Load all greenwash JSON files (excluding template)
gw_files = [f for f in glob.glob(f"{RAG_DIR}/greenwash_*.json") if "TEMPLATE" not in f]

if not gw_files:
    print("No greenwashing JSON files found yet.")
    print(f"Save Claude Projects outputs to: {RAG_DIR}/greenwash_TICKER.json")
    print("Then re-run this cell.")
else:
    print(f"Found {len(gw_files)} greenwashing assessments:")
    for f in gw_files:
        print(f"  {os.path.basename(f)}")

No greenwashing JSON files found yet.
Save Claude Projects outputs to: ../outputs/rag/greenwash_TICKER.json
Then re-run this cell.


In [6]:
DIMENSIONS = ["specificity", "metric", "baseline", "target", "time_horizon", "scope", "verification", "consistency"]
RATING_MAP = {"LOW": 0, "MED": 1, "HIGH": 2, "MISSING": 1}  # MISSING treated as MED risk

records = []
for fpath in gw_files:
    with open(fpath) as f:
        data = json.load(f)
    
    row = {
        "ticker": data.get("ticker"),
        "company_name": data.get("company_name"),
        "extraction_date": data.get("extraction_date"),
        "analyst_note": data.get("analyst_note", "")
    }
    
    dims = data.get("dimensions", {})
    high_count  = 0
    miss_count  = 0
    score_total = 0
    
    for dim in DIMENSIONS:
        info = dims.get(dim, {})
        rating = (info.get("rating") or "MISSING").upper()
        row[f"gw_{dim}_rating"]    = rating
        row[f"gw_{dim}_reasoning"] = info.get("reasoning", "")
        row[f"gw_{dim}_quote"]     = info.get("quote", "")
        row[f"gw_{dim}_page"]      = info.get("page", "")
        if rating == "HIGH":    high_count += 1
        if rating == "MISSING": miss_count += 1
        score_total += RATING_MAP.get(rating, 1)
    
    row["gw_high_count"]    = high_count
    row["gw_missing_count"] = miss_count
    row["gw_raw_score"]     = score_total  # 0 (best) to 16 (worst)
    row["gw_score_pct"]     = round(score_total / (len(DIMENSIONS) * 2) * 100, 1)  # 0â€“100%
    
    # Mandate exclusion rule: HIGH on >= 3 dimensions
    row["gw_exclude"]       = (high_count >= 3)
    # Watchlist: HIGH on exactly 2 dimensions
    row["gw_watchlist"]     = (high_count == 2)
    
    records.append(row)

if records:
    gw_df = pd.DataFrame(records)
    print(f"Greenwashing assessments imported: {len(gw_df)} companies")
    print(f"\nExclusion decisions:")
    print(f"  EXCLUDE (HIGH >= 3):  {gw_df['gw_exclude'].sum()} companies")
    print(f"  WATCHLIST (HIGH = 2): {gw_df['gw_watchlist'].sum()} companies")
    print(f"  PASS:                 {(~gw_df['gw_exclude'] & ~gw_df['gw_watchlist']).sum()} companies")
else:
    gw_df = pd.DataFrame()
    print("No records yet.")

No records yet.


## Step 5 — Greenwashing scorecard

In [7]:
if not gw_df.empty:
    rating_cols = [f"gw_{d}_rating" for d in DIMENSIONS]
    
    print("GREENWASHING SCORECARD")
    print("=" * 80)
    print(f"{'Company':<30} {'HIGH':>4} {'MED':>4} {'MISS':>4} {'Score':>6} {'Decision':<12}")
    print("-" * 80)
    
    for _, row in gw_df.sort_values("gw_score_pct", ascending=False).iterrows():
        ratings = [row[c] for c in rating_cols]
        n_high = ratings.count("HIGH")
        n_med  = ratings.count("MED")
        n_miss = ratings.count("MISSING")
        
        decision = "EXCLUDE" if row["gw_exclude"] else ("WATCHLIST" if row["gw_watchlist"] else "PASS")
        name = str(row.get("company_name", row["ticker"]))[:28]
        print(f"  {name:<28} {n_high:>4} {n_med:>4} {n_miss:>4} {row['gw_score_pct']:>5.1f}% {decision:<12}")
    
    print("=" * 80)
    print("\nDimension-level RED FLAG frequency:")
    for dim in DIMENSIONS:
        col = f"gw_{dim}_rating"
        n_high = (gw_df[col] == "HIGH").sum()
        print(f"  {dim:<15} HIGH flags: {n_high}/{len(gw_df)}")
else:
    print("No data to display.")

No data to display.


## Step 6 — Verification tracking

In [8]:
if not gw_df.empty:
    total_extractions = len(gw_df) * len(DIMENSIONS)
    sample_size = max(1, round(total_extractions * 0.30))
    
    print(f"Total extractions:          {total_extractions}")
    print(f"30% sample to verify:       {sample_size} (random selection below)")
    print(f"100% verify if:             Company is on EXCLUDE or WATCHLIST list")
    
    # Generate random sample for manual verification
    np.random.seed(42)
    verify_records = []
    for _, row in gw_df.iterrows():
        for dim in DIMENSIONS:
            verify_records.append({
                "ticker": row["ticker"],
                "dimension": dim,
                "rating": row.get(f"gw_{dim}_rating", ""),
                "quote": str(row.get(f"gw_{dim}_quote", ""))[:80],
                "page": row.get(f"gw_{dim}_page", ""),
                "priority": "MANDATORY" if row["gw_exclude"] or row["gw_watchlist"] else "SAMPLE"
            })
    
    verify_df = pd.DataFrame(verify_records)
    mandatory = verify_df[verify_df["priority"] == "MANDATORY"]
    sample_pool = verify_df[verify_df["priority"] == "SAMPLE"]
    
    sample_drawn = sample_pool.sample(min(sample_size, len(sample_pool)), random_state=42)
    verification_list = pd.concat([mandatory, sample_drawn]).reset_index(drop=True)
    verification_list["verified_by"] = ""
    verification_list["verification_date"] = ""
    verification_list["verification_result"] = ""  # CONFIRMED / CORRECTED / NOT FOUND
    verification_list["corrected_rating"] = ""
    
    print(f"\nVerification list: {len(mandatory)} mandatory + {len(sample_drawn)} random sample = {len(verification_list)} total")
    verification_list[["ticker", "dimension", "rating", "page", "priority"]].head(15)
else:
    verification_list = pd.DataFrame()
    print("No data.")

No data.


## Step 7 — Save all outputs

In [9]:
if not gw_df.empty:
    os.makedirs("../outputs/scores", exist_ok=True)

    # Full greenwashing scorecard
    gw_path = f"../outputs/scores/greenwashing_scores_{TODAY}.csv"
    gw_df.to_csv(gw_path, index=False)
    print(f"Greenwashing scores saved:    {gw_path}")

    # Exclusion list from greenwashing
    excl_cols = ["ticker", "company_name", "gw_high_count", "gw_score_pct", "gw_exclude", "gw_watchlist", "analyst_note"]
    excl_path = f"../outputs/portfolio/greenwashing_exclusions_{TODAY}.csv"
    os.makedirs("../outputs/portfolio", exist_ok=True)
    gw_df[excl_cols].to_csv(excl_path, index=False)
    print(f"Exclusion list saved:         {excl_path}")

    # Verification checklist (RAG Operator fills this in)
    if not verification_list.empty:
        ver_path = f"../outputs/scores/verification_checklist_{TODAY}.csv"
        verification_list.to_csv(ver_path, index=False)
        print(f"Verification checklist saved: {ver_path}")
    
    print("\nAll greenwashing outputs saved.")
else:
    print("No data to save yet.")

No data to save yet.


In [10]:
# ── Portfolio change log ───────────────────────────────────────────────────────
# Appends greenwashing decisions to outputs/rag/portfolio_change_log.csv.
# This file is read by notebook 10 and notebook 12 for full traceability.
# Rows are appended (not overwritten) so re-runs are timestamped separately.

import csv
from datetime import datetime

CHANGE_LOG_PATH = "../outputs/rag/portfolio_change_log.csv"
CHANGE_LOG_FIELDS = [
    "timestamp", "bloomberg_ticker", "company_name",
    "previous_status", "new_status", "action_taken",
    "triggering_module", "triggering_reason",
    "supporting_evidence_summary", "source_document", "source_url",
    "evidence_confidence", "analyst_override", "override_reason",
    "replacement_company_if_any",
]

os.makedirs("../outputs/rag", exist_ok=True)

# Initialise file with headers if it does not yet exist
if not os.path.exists(CHANGE_LOG_PATH):
    with open(CHANGE_LOG_PATH, "w", newline="", encoding="utf-8") as _f:
        csv.DictWriter(_f, fieldnames=CHANGE_LOG_FIELDS).writeheader()

if not gw_df.empty:
    _ts = datetime.now().isoformat(timespec="seconds")
    _new_rows = []

    for _, _row in gw_df.iterrows():
        _ticker  = _row["ticker"]
        _company = _row.get("company_name", _ticker)
        _highs   = int(_row.get("gw_high_count", 0))
        _score   = _row.get("gw_score_pct", "")

        # Worst dimension = the one with the highest risk rating
        _worst = max(
            DIMENSIONS,
            key=lambda d: {"HIGH": 2, "MED": 1, "MISSING": 1, "LOW": 0}.get(
                str(_row.get(f"gw_{d}_rating", "")).upper(), 0
            ),
        ) if _highs > 0 else "none"

        if _row.get("gw_exclude"):
            _action = "INCLUDED → EXCLUDED"
            _reason = f"Greenwashing HIGH on {_highs}/8 dimensions (exclusion threshold: ≥3)"
            _new_st = "EXCLUDED"
        elif _row.get("gw_watchlist"):
            _action = "INCLUDED → WATCHLIST"
            _reason = f"Greenwashing HIGH on {_highs}/8 dimensions (watchlist threshold: 2)"
            _new_st = "WATCHLIST"
        else:
            _action = "NO_ACTION"
            _reason = f"Greenwashing HIGH on {_highs}/8 dimensions — below thresholds"
            _new_st = "INCLUDED"

        _note = str(_row.get("analyst_note", ""))[:100]

        _new_rows.append({
            "timestamp":                   _ts,
            "bloomberg_ticker":            _ticker,
            "company_name":                _company,
            "previous_status":             "INCLUDED",
            "new_status":                  _new_st,
            "action_taken":                _action,
            "triggering_module":           "09_greenwashing.ipynb",
            "triggering_reason":           _reason,
            "supporting_evidence_summary": (
                f"Score: {_score}%  |  Worst dimension: {_worst}"
                + (f"  |  Note: {_note}" if _note else "")
            ),
            "source_document":             f"outputs/rag/greenwash_{_ticker}.json",
            "source_url":                  "",
            "evidence_confidence":         "",
            "analyst_override":            "N",
            "override_reason":             "",
            "replacement_company_if_any":  "",
        })

    with open(CHANGE_LOG_PATH, "a", newline="", encoding="utf-8") as _f:
        csv.DictWriter(_f, fieldnames=CHANGE_LOG_FIELDS).writerows(_new_rows)

    _excl = sum(1 for r in _new_rows if "EXCLUDED" in r["action_taken"])
    _wtch = sum(1 for r in _new_rows if "WATCHLIST" in r["action_taken"])
    _none = len(_new_rows) - _excl - _wtch

    print(f"Portfolio change log: {len(_new_rows)} entries written → {CHANGE_LOG_PATH}")
    print(f"  EXCLUDED:  {_excl}")
    print(f"  WATCHLIST: {_wtch}")
    print(f"  NO ACTION: {_none}")
else:
    print("No greenwashing data — change log not updated")


No greenwashing data — change log not updated


## âœ… Notebook complete

You now have:
- An **8-dimension greenwashing scorecard** for every company assessed
- **Exclusion and watchlist decisions** based on mandate rules
- A **verification checklist** showing which extractions the RAG Operator must check manually

**For Q&A on greenwashing:**  
- *What is the most common failure?* â€” Verification (self-reported only) and Time Horizon (2050 net zero with no interim milestones) are typically the weakest dimensions across all companies.
- *How do you prevent hallucination?* â€” Every rating cites a direct quote with page number. MISSING means absent from the document, never inferred.
- *What is limited vs reasonable assurance?* â€” Limited assurance ("nothing has come to our attention") is weaker than reasonable assurance (positive conclusion). CSRD will require reasonable assurance from 2028.

**Next:** Open `10_human_review.ipynb` to log any manual overrides.